<a href="https://colab.research.google.com/github/hyunwook3910/bigdata/blob/main/week04_5695610_%EC%84%9D%ED%98%84%EC%9A%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4주차 과제
빅데이터프로그래밍 · 통계학과

**이름:**
**학번:**

## 제출 방법
모든 셀을 실행해 출력이 보이는 상태로 저장한 뒤 `week04_학번_이름.ipynb` 로 제출합니다.

GPU 문제는 `런타임 > 런타임 유형 변경 > T4 GPU` 로 바꾼 뒤 실행하세요.


In [37]:
import torch
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "/ device:", device)


torch 2.11.0+cu130 / device: cuda


## 문제 1. Tensor 5개 생성과 속성 확인표 (40점)
서로 다른 Tensor 5개를 만들고, 아래 항목을 담은 표를 출력하세요.

| 이름 | shape | dim | dtype | device |

각 Tensor를 왜 그렇게 만들었는지 주석으로 한 줄 적으세요.


In [38]:
# 답안
device="cuda" if torch.cuda.is_available() else "cpu"
#정수 직접 입력 후 1차원 텐서 생성
a=torch.tensor([10,20,30],dtype=torch.int32).to(device)

#실수 직접 입력 후 1차원 텐서 생성
b=torch.tensor([1.5,2.5,3.5],dtype=torch.float32).to(device)

#2행 3열 모두 0으로 채운 2차원 텐서 생성
c=torch.zeros(2,3).to(device)

#2행 3열에 무작위 값을 채운 2차원 텐서 생성
d=torch.randn(2,3).to(device)

#3행 2열을 모두 1로 채운 2차원 실수 텐서 생성
e=torch.ones(3,2).to(device)

tensors={
    "Tensor 1":a,
    "Tensor 2":b,
    "Tensor 3":c,
    "Tensor 4":d,
    "Tensor 5":e,
}

data=[]
for name,tensor in tensors.items():
  data.append({
      "이름":name,
      "shape":list(tensor.shape),
      "dim":tensor.ndim,
      "dtype":str(tensor.dtype),
      "device":str(tensor.device)
  })

result=pd.DataFrame(data)
display(result)

,이름,shape,dim,dtype,device
0,Tensor 1,[3],1,torch.int32,cuda:0
1,Tensor 2,[3],1,torch.float32,cuda:0
2,Tensor 3,"[2, 3]",2,torch.float32,cuda:0
3,Tensor 4,"[2, 3]",2,torch.float32,cuda:0
4,Tensor 5,"[3, 2]",2,torch.float32,cuda:0


## 문제 2. Tensor 연산 (20점)
(3, 4) 와 (4, 2) Tensor를 만들어 아래를 출력하세요.
- 행렬곱 결과와 그 shape
- 첫 번째 Tensor의 열별 평균과 행별 합계


In [39]:
# 답안

#(3,4)와 (4,2) Tensor 만들기
x1=torch.randn(3,4,device=device)
x2=torch.randn(4,2,device=device)

print(x1)
print(x2)

#행렬곱과 shape 출력
result=torch.matmul(x1,x2)
print("행렬곱 결과:",result)
print("행렬곱 결과 shape:",result.shape)

#첫 번째 Tensor의 열별 평균과 행별 합계
colmean=x1.mean(dim=0)
rowsum=x1.sum(dim=1)
print("열별평균:",colmean)
print("행별합계:",rowsum)

tensor([[ 0.6902,  1.2495,  1.6552, -1.3078],
        [-0.6017,  0.2979,  1.1985, -0.4940],
        [ 1.1253, -0.3365, -0.7125,  1.1208]], device='cuda:0')
tensor([[ 0.5059,  0.5893],
        [ 0.4761, -0.0348],
        [ 0.5709, -0.4434],
        [ 0.3363, -0.5962]], device='cuda:0')
행렬곱 결과: tensor([[ 1.4493,  0.4090],
        [ 0.3556, -0.6018],
        [ 0.3792,  0.3225]], device='cuda:0')
행렬곱 결과 shape: torch.Size([3, 2])
열별평균: tensor([ 0.4046,  0.4036,  0.7138, -0.2270], device='cuda:0')
행별합계: tensor([2.2871, 0.4007, 1.1971], device='cuda:0')


## 문제 3. CPU와 GPU 비교 (20점)
같은 크기의 행렬곱을 CPU와 GPU에서 각각 실행해 걸린 시간을 출력하고, 차이를 한두 줄로 설명하세요.
GPU를 쓸 수 없는 경우에는 CPU 결과와 그 이유를 적으세요.


In [44]:
# 답안
import torch
import time

def bench(device, n=5000, repeat=3):
a=torch.randn(n,n,device=device)
b=torch.randn(n,n,device=device)

_=a@b
if=device == "cuda":
  torch.cuda.synchronize()
  start=time.time()
  for _ in range(repeat):
    _ =a@b

  if device == "cuda":
    torch.cuda.synchronize()

    return (time.time()-start)/repeat

cpu_time=bench('cpu')
print(f"cpu행렬곱 시간:{cpu_time:.4f}초")

cpu행렬곱 시간: 2.0520초
GPU 행렬곱 평균 소요 시간:0.0755초
GPU가 CPU보다27.2배 빠르다.


## 문제 4. 자동미분 (20점)
아래 두 가지를 하세요.
1. $y = 2x^3 + 5x$ 에서 $x=3$ 일 때의 도함수를 자동미분으로 구하고, 손으로 구한 값과 비교하세요.
2. 자기가 만든 수식 하나에 대해 미분값을 구하고, 손 계산과 일치하는지 확인하세요.


In [50]:
# 답안
x=torch.tensor(3.0,requires_grad=True)
y=2*(x**3)+5*x
y.backward()
result=x.grad.item()

handresult=6*(3**2)+5

print(result)
print(handresult)
print(result==handresult)

x2=torch.tensor(2.0,requires_grad=True)
y2=3*(x2**2)+4*x2+1
y2.backward()
result2=x2.grad.item()
handresult2=6*2+4

print(result2)
print(handresult2)
print(result2==handresult2)

59.0
59
True
16.0
16
True
